# Initialization

In [ ]:
# Laborers
import os
import pandas as pd
import numpy as np
from thefuzz import process

# Image workers
import cv2 as cv
import tesserocr
from PIL import Image, ImageOps

api = tesserocr.PyTessBaseAPI(path = "C:/Program Files/Tesseract-OCR/tessdata")

# Main

## Image processing

In [ ]:
# Set all coordinates for cropping.
crop_coords = [
    (250, 350, 900, 450),
    (250, 600, 900, 700),
    (250, 925, 900, 1000),
    (250, 1125, 900, 1250),
    (250, 1425, 900, 1525),
    (250, 1650, 900, 1800)
]

In [ ]:
results = []

for result in os.listdir(r"cq_results"):
    # Summon and crop.
    img = Image.open(f"cq_results/{result}").convert("L")
    for coords in crop_coords:
        curr = img.crop(coords)
        curr = api.SetImage(curr)
        # OCR and append.
        results.append(api.GetUTF8Text())

In [136]:
results_df = pd.DataFrame({"drop" : results,
                           "level" : np.tile([6,5,4,3,2,1], int(len(results)/6))})

In [156]:
results_df

,drop,level
0,Soulstone Ticket: Major ~ a\n,6
1,Invoker * 1\n,5
2,Destructive Memory ~ 1\n,4
3,Invoker * 1\n,3
4,Invoker * 1\n,2
...,...,...
541,Invoker * 1\n,5
542,Destructive Memory ~ 1\n,4
543,Soulstone Ticket: Major ~ 1\n,3
544,Destructive Memory ~ 1\n,2


## String cleaning

In [ ]:
# Get names.
raw = list(set(results))

# Correct names.
names = list(("Lu Lingqi", "Kintaro's Axe", "Pipe Fox",
             "Invoker", "Revive Charm",
             "Soulstone Ticket: Moderate", "Soulstone Ticket: Major",
             "Mysterious Memory", "Destructive Memory", "Tender Memory",
             "Tyrannical Memory", "Demon King's Memory", "War God's Memory")
             )

In [ ]:
matches_list = []

# Process matches.
for correct in names:
    matches = pd.DataFrame(process.extract(correct, raw, limit = 30), 
                           columns = ["match", "prob"])
    # Reasonable first-pass cutoff.
    matches = matches[matches["prob"] >= 50]
    print(matches)
    matches_list.append(matches)

In [138]:
# Determine better match probabilities for Memory, Soulstone, and everything else.
match_memory = 70
match_soulstone = 87
match_else = 60

matches_cut_list = []

for matches in matches_list:
    if "Memory" in matches.iloc[1,0]:
        matches = matches[matches["prob"] >= match_memory]
    elif "Soulstone" in matches.iloc[1,0]:
        matches = matches[matches["prob"] >= match_soulstone]
    else:
         matches = matches[matches["prob"] >= match_else]
         # Discard mistaken magatama.
         matches = matches[~matches["match"].str.contains("Memory")]
    matches_cut_list.append(matches)

In [171]:
# Iterate across each match.
for i, entry in enumerate(matches_cut_list):
    results_df["drop"] = results_df["drop"].replace(list(entry["match"]), names[i])

results_df.to_csv("gcq_final.csv")